# The Agent Loop — Building the Core Execution Engine

## What This Notebook Teaches You

Every AI agent — from a simple chatbot to an autonomous researcher — is driven by the same fundamental pattern: **the agent loop**. In this notebook, you will build it from scratch.

By the end, you will be able to:

1. **Explain the universal agent loop** — the perceive → reason → act cycle that powers all agents
2. **Design agent state** — build an `AgentState` dataclass that tracks everything an agent needs
3. **Implement AgentLoop** — a complete, reusable agent loop class with step-by-step execution
4. **Parse structured output** — extract JSON actions from LLM responses with robust fallbacks
5. **Implement termination strategies** — know when and how an agent should stop
6. **Analyze agent behavior** — trace execution, measure costs, and diagnose problems

---

> **Prerequisites**: Complete Notebook 01 (Introduction to Agentic AI).
> **Runtime**: Google Colab with T4 GPU (free tier works).
> **Time**: ~45–60 minutes.

## Part 0: Environment Setup

We load **Qwen3-14B** with 4-bit quantization — the same model used throughout this course.

In [ ]:
# --- Google Colab Setup ---
# Install dependencies (run once per session)
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
import time
import json
import re
import math
import inspect
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any, Callable, Tuple, Union, get_type_hints
from collections import defaultdict
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Mount Google Drive for model caching (faster subsequent loads)
drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

# Load open-source model with 4-bit quantization (fits Colab T4 16GB GPU)
MODEL_NAME = "Qwen/Qwen3-14B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
print(f"  Vocabulary size: {len(tokenizer):,} tokens")

## 1. The Universal Agent Loop

### 1.1 — The Pattern Every Agent Shares

No matter how sophisticated an agent becomes — whether it uses tools, memory, multi-agent collaboration, or planning — at its core it runs **the same loop**:

```
┌──────────────────────────────────────────────┐
│            THE UNIVERSAL AGENT LOOP          │
│                                              │
│   ┌─────────┐                                │
│   │  START   │                                │
│   └────┬────┘                                │
│        ▼                                     │
│   ┌─────────────┐                            │
│   │ Initialize   │  ← Set goal, empty state  │
│   │ State        │                            │
│   └────┬────────┘                            │
│        ▼                                     │
│   ┌─────────────┐     Yes                    │
│   │ Should stop? ├──────────► Return answer  │
│   └────┬────────┘                            │
│        │ No                                  │
│        ▼                                     │
│   ┌─────────────┐                            │
│   │ Format       │  ← Build prompt from state│
│   │ Prompt       │                            │
│   └────┬────────┘                            │
│        ▼                                     │
│   ┌─────────────┐                            │
│   │ Call LLM     │  ← Generate next action   │
│   └────┬────────┘                            │
│        ▼                                     │
│   ┌─────────────┐                            │
│   │ Parse        │  ← Extract structured     │
│   │ Response     │    action from text        │
│   └────┬────────┘                            │
│        ▼                                     │
│   ┌─────────────┐                            │
│   │ Update       │  ← Record step in state   │
│   │ State        │                            │
│   └────┬────────┘                            │
│        │                                     │
│        └─────── Loop back to "Should stop?"  │
└──────────────────────────────────────────────┘
```

### 1.2 — Pseudocode

```python
def agent_loop(goal):
    state = initialize(goal)
    while not should_stop(state):
        prompt = format_prompt(state)
        response = llm(prompt)
        action = parse(response)
        state = update(state, action)
    return state.final_answer
```

This looks simple, but each step hides real complexity:
- **initialize**: What goes in the state? How much context do we keep?
- **should_stop**: Max steps? LLM self-termination? Convergence?
- **format_prompt**: How do we compress history? What system prompt?
- **parse**: What if the LLM returns malformed JSON? What fallback?
- **update**: Do we keep all history or summarize?

Let's tackle each one.

## 2. Designing Agent State

### 2.1 — What State Does an Agent Need?

An agent must track:

| Component | Purpose | Example |
|-----------|---------|---------|
| **Goal** | What the agent is trying to achieve | "Explain quantum entanglement" |
| **Messages** | Conversation history with the LLM | System prompt + user/assistant turns |
| **Steps** | Record of all actions taken | List of (thought, action, result) tuples |
| **Current step** | Position in the execution | Step 3 of max 10 |
| **Max steps** | Safety limit to prevent infinite loops | 10 |
| **Completion flag** | Whether the agent has finished | True/False |
| **Final answer** | The result to return | "Quantum entanglement is..." |
| **Metadata** | Timing, token counts, diagnostics | {"total_tokens": 2340, "elapsed": 12.3} |

### 2.2 — The AgentState Dataclass

We use Python's `@dataclass` with mutable defaults via `field(default_factory=...)`.

In [ ]:
@dataclass
class AgentState:
    """Complete state of an agent during execution."""
    goal: str = ""
    messages: List[Dict[str, str]] = field(default_factory=list)
    steps: List[Dict[str, Any]] = field(default_factory=list)
    current_step: int = 0
    max_steps: int = 10
    is_complete: bool = False
    final_answer: str = ""
    metadata: Dict[str, Any] = field(default_factory=lambda: {
        "start_time": None,
        "end_time": None,
        "total_tokens_estimated": 0,
        "step_times": [],
        "step_token_estimates": [],
    })

    def elapsed_time(self) -> float:
        """Total elapsed time in seconds."""
        if self.metadata["start_time"] is None:
            return 0.0
        end = self.metadata["end_time"] or time.time()
        return end - self.metadata["start_time"]

    def summary(self) -> str:
        """Human-readable summary of agent execution."""
        status = "COMPLETE" if self.is_complete else "IN PROGRESS"
        return (
            f"Agent State [{status}]\n"
            f"  Goal: {self.goal}\n"
            f"  Steps: {self.current_step}/{self.max_steps}\n"
            f"  Time: {self.elapsed_time():.1f}s\n"
            f"  Answer: {self.final_answer[:100]}{'...' if len(self.final_answer) > 100 else ''}"
        )

# Test it
state = AgentState(goal="What is 2+2?", max_steps=5)
print(state.summary())
print(f"\nFields: {[f.name for f in state.__dataclass_fields__.values()]}")

## 3. Building the AgentLoop Class

### 3.1 — Architecture

Our `AgentLoop` has five core methods:

| Method | Purpose |
|--------|---------|
| `__init__(max_steps, system_prompt)` | Configure the agent |
| `run(goal)` | Execute the full loop |
| `step(state)` | Execute one iteration |
| `_should_stop(state)` | Check termination conditions |
| `_format_prompt(state)` | Build the LLM prompt from current state |

### 3.2 — The System Prompt

We instruct the LLM to respond with JSON in one of two formats:

```json
{"action": "think", "thought": "I need to break this problem into parts..."}
{"action": "answer", "answer": "The final answer is 42."}
```

This gives the agent two modes: **deliberation** (think) and **commitment** (answer).

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a reasoning agent. You solve problems step by step.

On each turn, you MUST respond with EXACTLY one JSON object (no other text):

Option 1 - Think (reason about the problem):
{"action": "think", "thought": "your reasoning here"}

Option 2 - Answer (provide final answer):
{"action": "answer", "answer": "your final answer here"}

Rules:
- Use "think" to break down the problem, consider approaches, and work through steps.
- Use "answer" ONLY when you are confident you have the complete solution.
- Each "think" step should make progress toward the answer.
- Respond with ONLY the JSON object, no additional text.
"""

print("System prompt defined:")
print(AGENT_SYSTEM_PROMPT[:200] + "...")

### 3.3 — JSON Parsing with Fallback

LLMs don't always produce clean JSON. We need robust parsing.

In [ ]:
def parse_agent_response(text: str) -> Dict[str, Any]:
    """Parse agent JSON response with multiple fallback strategies."""
    text = text.strip()

    # Strategy 1: Direct JSON parse
    try:
        result = json.loads(text)
        if "action" in result:
            return result
    except json.JSONDecodeError:
        pass

    # Strategy 2: Extract JSON from markdown code blocks
    code_block = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if code_block:
        try:
            result = json.loads(code_block.group(1))
            if "action" in result:
                return result
        except json.JSONDecodeError:
            pass

    # Strategy 3: Find first {...} block in the text
    brace_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)
    if brace_match:
        try:
            result = json.loads(brace_match.group())
            if "action" in result:
                return result
        except json.JSONDecodeError:
            pass

    # Strategy 4: Look for answer-like patterns
    answer_match = re.search(r'(?:answer|final answer|result)[:\s]+(.+)', text, re.IGNORECASE)
    if answer_match:
        return {"action": "answer", "answer": answer_match.group(1).strip()}

    # Fallback: treat entire response as a thought
    return {"action": "think", "thought": text[:500]}

# Test the parser
test_cases = [
    '{"action": "think", "thought": "Let me consider..."}',
    'Here is my response: {"action": "answer", "answer": "42"}',
    'The answer is 42.',
    '```json\n{"action": "think", "thought": "Working on it..."}\n```',
    'Some random text without JSON',
]

for tc in test_cases:
    result = parse_agent_response(tc)
    print(f"Input:  {tc[:60]}...")
    print(f"Parsed: {result}")
    print()

### 3.4 — The Complete AgentLoop Class

In [ ]:
class AgentLoop:
    """A complete agent loop that reasons step-by-step to solve tasks."""

    def __init__(self, max_steps: int = 10, system_prompt: str = None):
        self.max_steps = max_steps
        self.system_prompt = system_prompt or AGENT_SYSTEM_PROMPT

    def run(self, goal: str) -> AgentState:
        """Execute the full agent loop for a given goal."""
        state = AgentState(goal=goal, max_steps=self.max_steps)
        state.metadata["start_time"] = time.time()
        state.messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Task: {goal}\n\nBegin by reasoning about this task."},
        ]

        print(f"{'='*60}")
        print(f"Agent starting: {goal}")
        print(f"{'='*60}")

        while not self._should_stop(state):
            state = self.step(state)

        state.metadata["end_time"] = time.time()
        print(f"\n{'='*60}")
        print(f"Agent finished in {state.current_step} steps ({state.elapsed_time():.1f}s)")
        print(f"Answer: {state.final_answer[:200]}")
        print(f"{'='*60}")
        return state

    def step(self, state: AgentState) -> AgentState:
        """Execute one step of the agent loop."""
        step_start = time.time()
        state.current_step += 1

        # Call LLM
        response_text = generate(state.messages, max_new_tokens=512, temperature=0.7)
        step_time = time.time() - step_start

        # Parse response
        parsed = parse_agent_response(response_text)

        # Record step
        step_record = {
            "step": state.current_step,
            "raw_response": response_text[:500],
            "parsed": parsed,
            "time": step_time,
        }
        state.steps.append(step_record)
        state.metadata["step_times"].append(step_time)

        # Estimate tokens (rough: 1 token ≈ 4 chars)
        est_tokens = len(response_text) // 4
        state.metadata["step_token_estimates"].append(est_tokens)
        state.metadata["total_tokens_estimated"] += est_tokens

        # Print step trace
        action = parsed.get("action", "unknown")
        print(f"\n  Step {state.current_step}: [{action.upper()}]")
        if action == "think":
            thought = parsed.get("thought", "")
            print(f"    {thought[:150]}{'...' if len(thought) > 150 else ''}")
        elif action == "answer":
            answer = parsed.get("answer", "")
            state.final_answer = answer
            state.is_complete = True
            print(f"    → {answer[:150]}{'...' if len(answer) > 150 else ''}")

        # Update messages for next turn
        state.messages.append({"role": "assistant", "content": response_text})
        if action == "think":
            state.messages.append({
                "role": "user",
                "content": "Continue reasoning. When you have the answer, use the answer action."
            })

        return state

    def _should_stop(self, state: AgentState) -> bool:
        """Check if the agent should stop."""
        if state.is_complete:
            return True
        if state.current_step >= state.max_steps:
            state.final_answer = "[Max steps reached without answer]"
            if state.steps:
                last = state.steps[-1]["parsed"]
                if last.get("thought"):
                    state.final_answer = f"[Incomplete] Last thought: {last['thought']}"
            state.is_complete = True
            return True
        return False

    def _format_prompt(self, state: AgentState) -> List[Dict[str, str]]:
        """Return the current message list (prompt is built incrementally)."""
        return state.messages

print("✓ AgentLoop class defined")

## 4. Running the Agent on Real Tasks

Let's test our agent loop on tasks of varying difficulty.

In [ ]:
agent = AgentLoop(max_steps=8)

# Task 1: Simple factual question
print("\n" + "▶"*20 + " TASK 1: Simple Question " + "◀"*20)
state1 = agent.run("What are the three states of matter? Give a brief explanation of each.")
print(f"\n{state1.summary()}")

In [ ]:
# Task 2: Reasoning task
print("\n" + "▶"*20 + " TASK 2: Reasoning " + "◀"*20)
state2 = agent.run(
    "A farmer has 17 sheep. All but 9 run away. How many sheep does the farmer have left? "
    "Think carefully before answering."
)
print(f"\n{state2.summary()}")

In [ ]:
# Task 3: Multi-step analysis
print("\n" + "▶"*20 + " TASK 3: Multi-Step Analysis " + "◀"*20)
state3 = agent.run(
    "Compare and contrast the advantages of solar energy vs wind energy. "
    "Consider: cost, reliability, environmental impact, and scalability. "
    "Provide a structured analysis."
)
print(f"\n{state3.summary()}")

In [ ]:
# Task 4: Complex synthesis
print("\n" + "▶"*20 + " TASK 4: Complex Synthesis " + "◀"*20)
state4 = agent.run(
    "Design a simple algorithm for sorting a list of numbers. "
    "Describe the algorithm step by step, analyze its time complexity, "
    "and explain when it would be preferred over other sorting methods."
)
print(f"\n{state4.summary()}")

## 5. Termination Strategies

When should an agent stop? This is a critical design decision. Too early and you get incomplete answers. Too late and you waste tokens on circular reasoning.

We implement four strategies as callable objects.

In [ ]:
# Strategy 1: Max Steps (simplest)
def stop_max_steps(state: AgentState, max_steps: int = 10) -> bool:
    """Stop after a fixed number of steps."""
    return state.current_step >= max_steps

# Strategy 2: LLM Self-Termination (trust the model)
def stop_self_terminate(state: AgentState, **kwargs) -> bool:
    """Stop when the LLM outputs an answer action."""
    if not state.steps:
        return False
    last_action = state.steps[-1]["parsed"].get("action", "")
    return last_action == "answer"

# Strategy 3: Convergence Detection (repeated similar thoughts)
def stop_convergence(state: AgentState, window: int = 3, threshold: float = 0.7, **kwargs) -> bool:
    """Stop when the last N thoughts are too similar (agent is going in circles)."""
    if len(state.steps) < window:
        return False

    recent_thoughts = []
    for step in state.steps[-window:]:
        thought = step["parsed"].get("thought", step["parsed"].get("answer", ""))
        recent_thoughts.append(set(thought.lower().split()))

    # Compute average Jaccard similarity
    similarities = []
    for i in range(len(recent_thoughts)):
        for j in range(i + 1, len(recent_thoughts)):
            if recent_thoughts[i] and recent_thoughts[j]:
                intersection = len(recent_thoughts[i] & recent_thoughts[j])
                union = len(recent_thoughts[i] | recent_thoughts[j])
                similarities.append(intersection / union if union > 0 else 0)

    avg_sim = sum(similarities) / len(similarities) if similarities else 0
    return avg_sim > threshold

# Strategy 4: Combined (production-ready)
def stop_combined(state: AgentState, max_steps: int = 10, window: int = 3,
                  threshold: float = 0.7, **kwargs) -> bool:
    """Combined strategy: any condition triggers stop."""
    if stop_self_terminate(state):
        return True
    if stop_max_steps(state, max_steps=max_steps):
        return True
    if stop_convergence(state, window=window, threshold=threshold):
        return True
    return False

print("✓ Four termination strategies defined:")
print("  1. stop_max_steps     — fixed step limit")
print("  2. stop_self_terminate — LLM says 'answer'")
print("  3. stop_convergence   — detects circular reasoning")
print("  4. stop_combined      — all of the above")

### 5.1 — Comparing Termination Strategies

Let's create a configurable agent that accepts a termination strategy.

In [ ]:
class ConfigurableAgent(AgentLoop):
    """Agent with pluggable termination strategy."""

    def __init__(self, max_steps: int = 10, stop_fn: Callable = None, **stop_kwargs):
        super().__init__(max_steps=max_steps)
        self.stop_fn = stop_fn or stop_combined
        self.stop_kwargs = stop_kwargs

    def _should_stop(self, state: AgentState) -> bool:
        if self.stop_fn(state, max_steps=self.max_steps, **self.stop_kwargs):
            if not state.is_complete:
                # Grab the best answer so far
                for step in reversed(state.steps):
                    p = step["parsed"]
                    if p.get("action") == "answer":
                        state.final_answer = p["answer"]
                        break
                else:
                    if state.steps:
                        last = state.steps[-1]["parsed"]
                        state.final_answer = last.get("thought", last.get("answer", "[No answer]"))
                state.is_complete = True
            return True
        return False

# Compare strategies on the same task
task = "What is the capital of France, and what is its population approximately?"
strategies = {
    "max_steps_3": (stop_max_steps, {"max_steps": 3}),
    "self_terminate": (stop_self_terminate, {"max_steps": 8}),
    "combined": (stop_combined, {"max_steps": 8, "window": 3, "threshold": 0.7}),
}

results = {}
for name, (fn, kwargs) in strategies.items():
    print(f"\n{'='*50}")
    print(f"Strategy: {name}")
    print(f"{'='*50}")
    agent = ConfigurableAgent(stop_fn=fn, **kwargs)
    state = agent.run(task)
    results[name] = {
        "steps": state.current_step,
        "time": state.elapsed_time(),
        "answer_len": len(state.final_answer),
    }

print(f"\n\n{'Strategy':<20} {'Steps':<8} {'Time (s)':<10} {'Answer Length':<15}")
print("-" * 55)
for name, r in results.items():
    print(f"{name:<20} {r['steps']:<8} {r['time']:<10.1f} {r['answer_len']:<15}")

## 6. Agent Execution Analysis

Let's analyze the traces from our earlier runs to understand agent behavior.

In [ ]:
def analyze_agent_run(state: AgentState) -> Dict[str, Any]:
    """Analyze an agent's execution trace."""
    if not state.steps:
        return {"error": "No steps recorded"}

    step_times = state.metadata["step_times"]
    step_tokens = state.metadata["step_token_estimates"]

    # Count action types
    action_counts = defaultdict(int)
    for step in state.steps:
        action_counts[step["parsed"].get("action", "unknown")] += 1

    analysis = {
        "total_steps": state.current_step,
        "total_time": state.elapsed_time(),
        "avg_step_time": sum(step_times) / len(step_times) if step_times else 0,
        "min_step_time": min(step_times) if step_times else 0,
        "max_step_time": max(step_times) if step_times else 0,
        "total_tokens_est": state.metadata["total_tokens_estimated"],
        "avg_tokens_per_step": (state.metadata["total_tokens_estimated"] / len(step_tokens)
                                if step_tokens else 0),
        "action_distribution": dict(action_counts),
        "think_ratio": action_counts.get("think", 0) / max(len(state.steps), 1),
    }
    return analysis

# Analyze all four tasks
for i, (label, state) in enumerate([
    ("Simple Question", state1),
    ("Reasoning", state2),
    ("Multi-Step", state3),
    ("Complex", state4),
], 1):
    analysis = analyze_agent_run(state)
    print(f"\nTask {i}: {label}")
    print(f"  Steps: {analysis['total_steps']} | Time: {analysis['total_time']:.1f}s")
    print(f"  Avg step time: {analysis['avg_step_time']:.1f}s")
    print(f"  Est. tokens: {analysis['total_tokens_est']}")
    print(f"  Actions: {analysis['action_distribution']}")
    print(f"  Think ratio: {analysis['think_ratio']:.0%}")

### 6.1 — Step-by-Step Trace Visualization

In [ ]:
def print_trace(state: AgentState):
    """Print a formatted execution trace."""
    print(f"\n{'='*60}")
    print(f"EXECUTION TRACE: {state.goal[:50]}")
    print(f"{'='*60}")

    for step in state.steps:
        s = step["step"]
        action = step["parsed"].get("action", "?")
        t = step["time"]
        symbol = "💭" if action == "think" else "✅" if action == "answer" else "❓"

        print(f"\n  {symbol} Step {s} ({t:.1f}s) [{action.upper()}]")
        if action == "think":
            thought = step["parsed"].get("thought", "")
            # Word-wrap at 70 chars
            words = thought.split()
            lines = []
            current = "    "
            for w in words:
                if len(current) + len(w) + 1 > 74:
                    lines.append(current)
                    current = "    " + w
                else:
                    current += " " + w if current.strip() else "    " + w
            if current.strip():
                lines.append(current)
            print("\n".join(lines))
        elif action == "answer":
            print(f"    → {step['parsed'].get('answer', '')[:200]}")

    print(f"\n  Total: {state.current_step} steps, {state.elapsed_time():.1f}s")
    print(f"{'='*60}")

# Show trace for the most interesting run
print_trace(state3)

## 7. The Agent as a Finite State Machine

We can formalize our agent loop as a **finite state machine (FSM)**:

```
                    ┌─────────────┐
                    │    IDLE      │
                    │  (no goal)   │
                    └──────┬──────┘
                           │ run(goal)
                           ▼
                    ┌─────────────┐
                    │  THINKING    │◄──────┐
                    │  (LLM call)  │       │
                    └──────┬──────┘       │
                           │              │
                    ┌──────▼──────┐       │
                    │   PARSING    │       │
                    │ (extract     │       │
                    │  action)     │       │
                    └──────┬──────┘       │
                           │              │
                     ┌─────┴─────┐        │
                     │           │        │
              action=think  action=answer  │
                     │           │        │
                     ▼           ▼        │
              ┌──────────┐ ┌──────────┐   │
              │ REASONING│ │ ANSWERING│   │
              │ (update  │ │ (set     │   │
              │  state)  │ │  final)  │   │
              └────┬─────┘ └────┬─────┘   │
                   │            │         │
                   │ continue   │ done    │
                   └────────────│─────────┘
                                ▼
                         ┌──────────┐
                         │ COMPLETE  │
                         │ (return)  │
                         └──────────┘
```

Each state represents a phase of the agent's execution. The transitions are deterministic — only the LLM's output (think vs answer) creates branching.

In [ ]:
# Let's implement the FSM view and track state transitions
def extract_state_transitions(state: AgentState) -> List[Tuple[str, str]]:
    """Extract state transitions from an agent run."""
    transitions = [("IDLE", "THINKING")]

    for step in state.steps:
        action = step["parsed"].get("action", "unknown")
        if action == "think":
            transitions.append(("THINKING", "REASONING"))
            transitions.append(("REASONING", "THINKING"))
        elif action == "answer":
            transitions.append(("THINKING", "ANSWERING"))
            transitions.append(("ANSWERING", "COMPLETE"))
        else:
            transitions.append(("THINKING", "UNKNOWN"))

    # If ended without answer
    if not state.is_complete or state.final_answer.startswith("["):
        if transitions[-1][1] != "COMPLETE":
            transitions.append((transitions[-1][1], "TIMEOUT"))

    return transitions

# Show transitions for each run
for label, state in [("Simple", state1), ("Reasoning", state2),
                      ("Multi-Step", state3), ("Complex", state4)]:
    transitions = extract_state_transitions(state)
    chain = " → ".join([t[0] for t in transitions] + [transitions[-1][1]])
    print(f"{label}:")
    print(f"  {chain}")
    print(f"  ({len(transitions)} transitions)")
    print()

## 8. Quality vs. Steps — How Much Thinking Is Enough?

A key question: does more thinking always produce better answers? Let's test with the same task at different step limits.

In [ ]:
task = (
    "Explain the concept of recursion in programming. "
    "Include a definition, an example, and common pitfalls."
)

step_limits = [2, 4, 6, 8]
quality_results = []

for limit in step_limits:
    print(f"\n{'─'*40}")
    print(f"Running with max_steps={limit}")
    print(f"{'─'*40}")
    agent = AgentLoop(max_steps=limit)
    state = agent.run(task)
    quality_results.append({
        "max_steps": limit,
        "actual_steps": state.current_step,
        "time": state.elapsed_time(),
        "answer_length": len(state.final_answer),
        "answer_preview": state.final_answer[:100],
    })

print(f"\n\n{'Max Steps':<12} {'Used':<8} {'Time':<10} {'Answer Len':<12} {'Preview':<50}")
print("─" * 92)
for r in quality_results:
    print(f"{r['max_steps']:<12} {r['actual_steps']:<8} {r['time']:<10.1f} "
          f"{r['answer_length']:<12} {r['answer_preview'][:50]}...")

## 9. Key Takeaways

### What We Built
We implemented a complete agent loop from scratch:
- **AgentState** — a dataclass tracking everything the agent needs
- **AgentLoop** — the core execution engine with step/run methods
- **JSON Parser** — robust extraction with 4 fallback strategies
- **Termination Strategies** — 4 approaches (max steps, self-terminate, convergence, combined)

### Core Insights

1. **The agent loop is a software pattern**, not magic. It's a while loop with state management.

2. **State design is the hardest part.** What you track determines what the agent can do. Messages grow without bound — in production you need summarization or sliding windows.

3. **Parsing is fragile.** LLMs don't reliably produce JSON. Always have fallbacks. Never trust raw output.

4. **Termination is a design choice.** No single strategy is best:
   - Max steps: safe but rigid
   - Self-termination: flexible but can hallucinate "done"
   - Convergence: smart but needs tuning
   - Combined: most robust

5. **More steps ≠ better answers.** There's a sweet spot. Beyond it, agents often go in circles or overthink.

### What's Next
In the next notebook, we add **tools** — giving the agent the ability to execute code, look up facts, and interact with the world. The agent loop stays the same; we just add a new action type.

---

```
Notebook 02 Complete — The Agent Loop
From simple loop to production engine
```